# Day 2: Parse DataCite (Offline)

DataCite is commonly used to describe datasets (especially when a DOI is involved). In this notebook we parse a local DataCite XML file for the teaching dataset.

### Colab tip
If the repository path is not available, upload `climate_dataset_datacite.xml`; this notebook automatically falls back to `/content/climate_dataset_datacite.xml`.


In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd

# ---- Notebook-local parser (Colab/Jupyter friendly) ----
def _local_name(tag: str) -> str:
    return tag.split('}', 1)[1] if '}' in tag else tag

def parse_datacite_xml(xml_path):
    root = ET.parse(xml_path).getroot()

    identifier = None
    creators = []
    title = None
    publisher = None
    publication_year = None
    resource_type = None
    description = None

    for el in root.iter():
        name = _local_name(el.tag)
        txt = (el.text or '').strip()
        if name == 'identifier' and txt and identifier is None:
            identifier = txt
        elif name == 'creatorName' and txt:
            creators.append(txt)
        elif name == 'title' and txt and title is None:
            title = txt
        elif name == 'publisher' and txt and publisher is None:
            publisher = txt
        elif name == 'publicationYear' and txt and publication_year is None:
            publication_year = txt
        elif name == 'resourceType' and resource_type is None:
            resource_type = el.attrib.get('resourceTypeGeneral') or txt
        elif name == 'description' and txt and description is None:
            description = txt

    return {
        'standard': 'DataCite',
        'identifier': identifier,
        'creator': creators,
        'title': title,
        'publisher': publisher,
        'publicationYear': publication_year,
        'resourceTypeGeneral': resource_type,
        'description': description,
    }

# Path strategy for local repo or Colab upload
default_path = Path('../../data/metadata/climate_dataset_datacite.xml')
dcite_path = default_path if default_path.exists() else Path('/content/climate_dataset_datacite.xml')

datacite_meta = parse_datacite_xml(dcite_path)
datacite_meta

In [ ]:
rows = [
    ("identifier", datacite_meta.get("identifier")),
    ("creators", "; ".join(datacite_meta.get("creator") or [])),
    ("title", datacite_meta.get("title")),
    ("publisher", datacite_meta.get("publisher")),
    ("publicationYear", datacite_meta.get("publicationYear")),
    ("resourceTypeGeneral", datacite_meta.get("resourceTypeGeneral")),
    ("description", datacite_meta.get("description")),
]

table = pd.DataFrame(rows, columns=["field", "value"])
table

## Teaching interpretation
- DataCite emphasizes citation-style fields such as `identifier`, `creators`, and `publicationYear`.
- These fields support FAIR Findability and Reusability because they help people cite and understand the dataset in a consistent way.
